<a href="https://www.kaggle.com/code/mh0386/spam-detection?scriptVersionId=138183938" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [3]:
from kagglehub import dataset_download
import tensorflow as tf
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Dense, Embedding

In [6]:
data = pl.read_csv(f'{dataset_download("team-ai/spam-text-message-classification")}/SPAM text message 20170820 - Data.csv')
data

Category,Message
str,str
"""ham""","""Go until jurong point, crazy..…"
"""ham""","""Ok lar... Joking wif u oni..."""
"""spam""","""Free entry in 2 a wkly comp to…"
"""ham""","""U dun say so early hor... U c …"
"""ham""","""Nah I don't think he goes to u…"
…,…
"""spam""","""This is the 2nd time we have t…"
"""ham""","""Will ü b going to esplanade fr…"
"""ham""","""Pity, * was in mood for that. …"


In [ ]:
data.groupby('Category').count()

In [ ]:
ham_msg = data.filter(pl.col('Category') == 'ham')
spam_msg = data.filter(pl.col('Category') == 'spam')

In [ ]:
#randomly taking data from ham_ msg
ham_msg = ham_msg.sample(n=len(spam_msg))

In [ ]:
print(ham_msg.shape, spam_msg.shape)

In [ ]:
data

In [ ]:
balanced_data = ham_msg.vstack(spam_msg)
balanced_data = balanced_data.with_columns(
    pl.when(pl.col("Category") == 'spam').then(1).otherwise(0).alias('Category')
)

In [ ]:
balanced_data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    balanced_data['Message'],
    balanced_data['Category'],
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train = np.array(X_train, dtype=np.chararray)
X_test = np.array(X_test, dtype=np.chararray)
X_train

In [ ]:
vocab_size = 10000
sequence_length = 100

# Create the layer.
vectorized_layer = TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length
)

X_test = tf.expand_dims(X_test, -1)  # add a new dimension
X_test = tf.squeeze(X_test, axis=1)

X_train = tf.expand_dims(X_train, -1)  # add a new dimension
X_train = tf.squeeze(X_train, axis=1)

vectorized_layer.adapt(X_test)
vectorized_X_test = vectorized_layer(X_test)

vectorized_layer.adapt(X_train)
vectorized_X_train = vectorized_layer(X_train)

In [ ]:
Y_train = np.array(Y_train, dtype=np.int64)
Y_test = np.array(Y_test, dtype=np.int64)

# model

In [ ]:
model = tf.keras.models.Sequential(
    [
        Embedding(vocab_size + 1, 100, input_length=sequence_length),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ]
)

In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'],
    optimizer='adam'
)

In [ ]:
history = model.fit(
    vectorized_X_train,
    Y_train,
    epochs=100,
    validation_split=0.2,
    batch_size=10
)

In [ ]:
model.evaluate(
    vectorized_X_test,
    Y_test
)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'valid'])

In [ ]:
predict_msg = [
    "Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",
    "Ok lar... Joking wif u oni...",
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"]

In [ ]:
vectorized_layer.adapt(predict_msg)
vectorized_predict_msg = vectorized_layer(predict_msg)
model.predict(vectorized_predict_msg)